<a href="https://colab.research.google.com/github/kanchanr8595-create/GeoPulse-Hyper-Local-Retail-Mobility-Analytics/blob/Dev_akshaya/Python/Data_Validation_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

store=pd.read_csv("/content/GeoPulse_Store_Locations.csv")
geo=pd.read_csv("/content/GeoPulse_500_Synthetic_geo_Pings.csv")

In [3]:
print("geo Dataset Shape:",geo.shape)
print("Store Dataset shape:",store.shape)

print("\ngeo Columns:")
print(geo.columns.tolist())

print("\nStore Columns:")
print(store.columns.tolist())

GPS Dataset Shape: (500, 15)
Store Dataset shape: (3, 6)

GPS Columns:
['ping_id', 'device_id', 'latitude', 'longitude', 'timestamp', 'activity_area', 'day_of_week', 'hour', 'daypart', 'movement_mode', 'gps_accuracy_m', 'is_anonymized', 'nearest_store_id', 'nearest_store_distance_m', 'within_500m_of_nearest_store']

Store Columns:
['store_id', 'store_name', 'store_status', 'latitude', 'longitude', 'catchment_radius_m']


In [5]:
print("geo DATA -MISSING VALUES")
print(geo.isnull().sum())

print("STORE DATA MISSING VALUES")
print(store.isnull().sum())



GPS DATA -MISSING VALUES
ping_id                         0
device_id                       0
latitude                        0
longitude                       0
timestamp                       0
activity_area                   0
day_of_week                     0
hour                            0
daypart                         0
movement_mode                   0
gps_accuracy_m                  0
is_anonymized                   0
nearest_store_id                0
nearest_store_distance_m        0
within_500m_of_nearest_store    0
dtype: int64
STORE DATA MISSING VALUES
store_id              0
store_name            0
store_status          0
latitude              0
longitude             0
catchment_radius_m    0
dtype: int64


In [9]:
print("\ngeo Duplicate rows:",geo.duplicated().sum())
print("\nStore Duplicate rows:",store.duplicated().sum())

geo_duplicate_keys=geo.duplicated(
    subset=["device_id","timestamp"],keep=False
)
print("\nRepeated device_id and Timestamp:")
print(geo[geo_duplicate_keys])


if "store_id" in store.columns:
  print("\nDuplicate Store IDs:")
  print(store[store.duplicated(
      subset=["store_id"],keep=False
  )])





GPS Duplicate rows: 0

Store Duplicate rows: 0

Repeated DeviceID and Timestamp:
Empty DataFrame
Columns: [ping_id, device_id, latitude, longitude, timestamp, activity_area, day_of_week, hour, daypart, movement_mode, gps_accuracy_m, is_anonymized, nearest_store_id, nearest_store_distance_m, within_500m_of_nearest_store]
Index: []

Duplicate Store IDs:
Empty DataFrame
Columns: [store_id, store_name, store_status, latitude, longitude, catchment_radius_m]
Index: []


In [12]:
print("Missing device_id:",
      geo["device_id"].isnull().sum())

print("blank device_id")
geo["device_id"] = geo["device_id"].astype("string").str.strip()

invalid_device = (
    geo["device_id"].isna() |
    (geo["device_id"] == "")
)

print("Invalid device_id Records:")
print(geo[invalid_device])

print("Unique devices")
print("\nTotal Unique Devices:",
      geo["device_id"].nunique())

Missing DeviceID: 0
blank DeviceID
Invalid DeviceID Records:
Empty DataFrame
Columns: [ping_id, device_id, latitude, longitude, timestamp, activity_area, day_of_week, hour, daypart, movement_mode, gps_accuracy_m, is_anonymized, nearest_store_id, nearest_store_distance_m, within_500m_of_nearest_store]
Index: []
Unique devices

Total Unique Devices: 118


In [28]:
geo["latitude"] = pd.to_numeric(
    geo["latitude"], errors="coerce"
)

geo["longitude"] = pd.to_numeric(
    geo["longitude"], errors="coerce"
)

store["latitude"] = pd.to_numeric(
    store["latitude"], errors="coerce"
)

store["longitude"] = pd.to_numeric(
    store["longitude"], errors="coerce"
)

# gps coordinate validation
invalid_gps_coordinates = geo[
    geo["latitude"].isna() |
    geo["longitude"].isna() |
    ~geo["latitude"].between(-90, 90) |
    ~geo["longitude"].between(-180, 180)
]

print("Invalid geo Coordinates:",
      len(invalid_gps_coordinates))

print(invalid_gps_coordinates)

# Store coordinate validation
invalid_store_coordinates = store[
    store["latitude"].isna() |
    store["longitude"].isna() |
    ~store["latitude"].between(-90, 90) |
    ~store["longitude"].between(-180, 180)
]

print("\nInvalid Store Coordinates:",
      len(invalid_store_coordinates))

print(invalid_store_coordinates)

Invalid geo Coordinates: 0
Empty DataFrame
Columns: [ping_id, device_id, latitude, longitude, timestamp, activity_area, day_of_week, hour, daypart, movement_mode, gps_accuracy_m, is_anonymized, nearest_store_id, nearest_store_distance_m, within_500m_of_nearest_store]
Index: []

Invalid Store Coordinates: 0
Empty DataFrame
Columns: [store_id, store_name, store_status, latitude, longitude, catchment_radius_m]
Index: []


In [26]:
geo["timestamp"] = pd.to_datetime(
    geo["timestamp"],
    errors="coerce"
)

invalid_timestamp = geo[
    geo["timestamp"].isna()
]

print("Invalid Timestamp Records:")
print(invalid_timestamp)

print("\nTotal Invalid Timestamps:",
      len(invalid_timestamp))

# Check chronological order by device
gps_valid_time = geo.dropna(
    subset=["device_id", "timestamp"]
).copy()

gps_valid_time = gps_valid_time.sort_values(
    ["device_id", "timestamp"]
)

print("\nRepeated Device-Time Combinations:",
      gps_valid_time.duplicated(
          subset=["device_id", "timestamp"]
      ).sum())

print("\nEarliest Timestamp:",
      gps_valid_time["timestamp"].min())

print("Latest Timestamp:",
      gps_valid_time["timestamp"].max())

Invalid Timestamp Records:
Empty DataFrame
Columns: [ping_id, device_id, latitude, longitude, timestamp, activity_area, day_of_week, hour, daypart, movement_mode, gps_accuracy_m, is_anonymized, nearest_store_id, nearest_store_distance_m, within_500m_of_nearest_store]
Index: []

Total Invalid Timestamps: 0

Repeated Device-Time Combinations: 0

Earliest Timestamp: 2026-10-01 00:09:03
Latest Timestamp: 2026-10-07 22:26:42


In [20]:
# Preserve original row order
gps_ordered = geo.copy()

gps_ordered["timestamp"] = pd.to_datetime(
    gps_ordered["timestamp"],
    errors="coerce"
)

gps_ordered["Previous_Timestamp"] = (
    gps_ordered.groupby("device_id")["timestamp"]
    .shift(1)
)

backward_timestamps = gps_ordered[
    gps_ordered["timestamp"] <
    gps_ordered["Previous_Timestamp"]
]

print("Backward Timestamp Records:")
print(backward_timestamps)

print("\nTotal Backward Timestamp Records:",
      len(backward_timestamps))

Backward Timestamp Records:
       ping_id   device_id   latitude  longitude           timestamp  \
40   PING_0041  DEVICE_017  28.466790  77.065455 2026-10-02 17:15:03   
42   PING_0043  DEVICE_102  28.460990  77.093907 2026-10-04 09:32:57   
51   PING_0052  DEVICE_052  28.497957  77.086295 2026-10-01 16:01:41   
54   PING_0055  DEVICE_018  28.479477  77.074213 2026-10-02 02:14:27   
55   PING_0056  DEVICE_035  28.457166  77.091536 2026-10-01 00:09:03   
..         ...         ...        ...        ...                 ...   
491  PING_0492  DEVICE_020  28.480143  77.073694 2026-10-06 19:22:43   
492  PING_0493  DEVICE_099  28.475577  77.071888 2026-10-02 19:25:43   
495  PING_0496  DEVICE_103  28.490724  77.079141 2026-10-01 18:12:15   
497  PING_0498  DEVICE_052  28.493715  77.090722 2026-10-02 22:28:19   
498  PING_0499  DEVICE_076  28.488110  77.080536 2026-10-02 09:39:17   

        activity_area day_of_week  hour       daypart   movement_mode  \
40          SECTOR_29      Friday 

In [25]:


quality_report = pd.DataFrame({
    "Validation Check": [
        "Missing Values",
        "Completely Duplicate Rows",
        "Invalid DeviceID",
        "Invalid GPS Coordinates",
        "Invalid Store Coordinates",
        "Invalid Timestamps",
        "Backward Timestamps",
        "Repeated Device-Time Combinations"
    ],

    "GPS Dataset": [
        geo.isnull().sum().sum(),
        geo.duplicated().sum(),
        invalid_device.sum(),
        len(invalid_gps_coordinates),
        "Not Applicable",
        len(invalid_timestamp),
        len(backward_timestamps),
        gps_valid_time.duplicated(
            subset=["device_id", "timestamp"]
        ).sum()
    ],

    "Store Dataset": [
        store.isnull().sum().sum(),
        store.duplicated().sum(),
        "Not Applicable",
        "Not Applicable",
        len(invalid_store_coordinates),
        "Not Applicable",
        "Not Applicable",
        "Not Applicable"
    ]
})

print(quality_report.to_string(index=False))


quality_report.to_csv(
    "GeoPulse_Data_Quality_Report.csv",
    index=False
)

print("\nData Quality Report Exported Successfully!")

                 Validation Check    GPS Dataset  Store Dataset
                   Missing Values              0              0
        Completely Duplicate Rows              0              0
                 Invalid DeviceID              0 Not Applicable
          Invalid GPS Coordinates              0 Not Applicable
        Invalid Store Coordinates Not Applicable              0
               Invalid Timestamps              0 Not Applicable
              Backward Timestamps            183 Not Applicable
Repeated Device-Time Combinations              0 Not Applicable

Data Quality Report Exported Successfully!
